In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import GradientBoostingRegressor

# ---------------- LOAD DATA ----------------
df = pd.read_csv(r"D:\AI_Based_Smart_Food_Waste_Reduction_System\dataset_creating\food_waste_raw_1000.csv")
df["date"] = pd.to_datetime(df["date"])

# ---------------- PRODUCT LIST ----------------
products = df["product"].unique()

print("Products found:", products)

# ---------------- FUNCTION ----------------
def create_lag_features(data, lag=7):
    df_lag = data.copy()
    for i in range(1, lag + 1):
        df_lag[f"lag_{i}"] = df_lag["predicted_demand"].shift(i)
    return df_lag

# ---------------- TRAIN LOOP ----------------
for product in products:
    print(f"\n🚀 Training model for: {product}")

    product_df = df[df["product"] == product]

    daily = product_df.groupby("date")["predicted_demand"].mean().reset_index()
    daily = daily.sort_values("date")

    # Skip if not enough data
    if len(daily) < 10:
        print(f"⚠️ Skipping {product} (not enough data)")
        continue

    # Time features
    daily["day_of_week"] = daily["date"].dt.weekday
    daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)

    # Lag features
    lag = 7
    lag_df = create_lag_features(daily, lag)
    lag_df = lag_df.dropna()

    if lag_df.empty:
        print(f"⚠️ Skipping {product} (lag issue)")
        continue

    # Features
    X = lag_df.drop(["date", "predicted_demand"], axis=1)
    y = lag_df["predicted_demand"]

    # Train model
    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X, y)

    # Save model
    joblib.dump(model, f"forecasting_model/gbr_{product}.pkl")

    # Save context
    last_values = daily["predicted_demand"].values[-lag:].tolist()
    last_date = daily["date"].iloc[-1]

    joblib.dump(last_values, f"forecasting_model/last_values_{product}.pkl")
    joblib.dump(last_date, f"forecasting_model/last_date_{product}.pkl")
    joblib.dump(X.columns.tolist(), f"forecasting_model/features_{product}.pkl")

    print(f"✅ Saved model for {product}")

print("\n🎉 ALL MODELS TRAINED SUCCESSFULLY!")

Products found: <StringArray>
['Chicken', 'Bread', 'Milk', 'Banana', 'Rice', 'Eggs', 'Tomato', 'Apple']
Length: 8, dtype: str

🚀 Training model for: Chicken
✅ Saved model for Chicken

🚀 Training model for: Bread
✅ Saved model for Bread

🚀 Training model for: Milk
✅ Saved model for Milk

🚀 Training model for: Banana
✅ Saved model for Banana

🚀 Training model for: Rice
✅ Saved model for Rice

🚀 Training model for: Eggs
✅ Saved model for Eggs

🚀 Training model for: Tomato
✅ Saved model for Tomato

🚀 Training model for: Apple
✅ Saved model for Apple

🎉 ALL MODELS TRAINED SUCCESSFULLY!
